# Album Classifier - SageMaker Training & Deployment

This notebook trains and deploys your album classification model.

## Prerequisites

1. ✅ Code and data uploaded to S3 (run `./prepare_for_studio.sh <bucket-name>`)
2. ✅ SageMaker Studio domain created
3. ✅ Update `BUCKET_NAME` below with your bucket name

## Step 1: Install Dependencies

In [ ]:
# Studio ships SageMaker v3 split packages; this project needs v2 (sagemaker.pytorch).
!pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops sagemaker-telemetry 2>/dev/null || true
!pip install --force-reinstall 'sagemaker==2.232.2' 'boto3>=1.35.0'

## Step 2: Configuration

**IMPORTANT**: Update `BUCKET_NAME` with your actual S3 bucket name!

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch, PyTorchModel
import boto3
import json

# ============================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================
BUCKET_NAME = 'amazon-sagemaker-993268716868-us-east-2-3njxscvwqvqyi1'
REGION = 'us-east-2'
ENDPOINT_NAME = 'album-classifier'

# Path to cloned repo in Studio (adjust if yours differs)
REPO_DIR = '/home/sagemaker-user/discogs-sagemaker'
BACKEND_DIR = f'{REPO_DIR}/backend'

# SDK 2.232.x image maps differ: training supports 2.4, inference only up to 2.3
PYTORCH_TRAIN_VERSION = '2.4.0'
PYTORCH_INFER_VERSION = '2.3.0'
TRAINING_INSTANCE_TYPE = 'ml.c4.xlarge'  # match your Service Quotas approval
PROCESSING_INSTANCE_TYPE = 'ml.c4.xlarge'  # processing job for build_data
ENDPOINT_INSTANCE_TYPE = 'ml.m5.large'
USE_SPOT_TRAINING = True  # set False for on-demand
RELEASE_COUNT = 50  # albums to parse/enrich/download
DATA_S3_PREFIX = 'data'

# Get execution role (automatically detected in Studio)
role = sagemaker.get_execution_role()
print(f"✓ Using role: {role}")

# Verify bucket access
s3_client = boto3.client('s3', region_name=REGION)
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"✓ Bucket '{BUCKET_NAME}' is accessible")
    
    # List files
    print("\nS3 Contents:")
    print(f"  Data: s3://{BUCKET_NAME}/data/")
    print(f"  Code: s3://{BUCKET_NAME}/code/")
except Exception as e:
    print(f"✗ Error accessing bucket: {e}")
    print("Make sure the bucket exists and your role has S3 permissions")

## Step 3: Build dataset (SageMaker Processing Job)

Runs on a **separate** instance — keeps going if you stop the Space.

1. Set `DISCOGS_TOKEN` in the cell below.
2. Launch processing job (`wait=False`).
3. Monitor in console or run the status cell.
4. Output lands in `s3://BUCKET/data/` (manifest + images).

Start with `RELEASE_COUNT = 50`, then increase once it works.

In [ ]:
import os
from pathlib import Path

# Discogs token — passed to the processing job (not written to disk)
# https://www.discogs.com/settings/developers → Generate new token
DISCOGS_TOKEN = os.environ.get('DISCOGS_USER_TOKEN', 'PASTE_YOUR_TOKEN_HERE')

if DISCOGS_TOKEN == 'PASTE_YOUR_TOKEN_HERE':
    print('⚠ Set DISCOGS_TOKEN before launching the processing job')
else:
    print('✓ DISCOGS_TOKEN set')

In [ ]:
from sagemaker.processing import ScriptProcessor, ProcessingOutput
from datetime import datetime
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.exists():
    raise FileNotFoundError(
        f'Clone repo first: git clone https://github.com/normtronics/discogs-sagemaker.git {REPO_DIR}'
    )

if DISCOGS_TOKEN == 'PASTE_YOUR_TOKEN_HERE':
    raise ValueError('Set DISCOGS_TOKEN in the cell above')

sess = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
image_uri = f'763104351884.dkr.ecr.{REGION}.amazonaws.com/pytorch-training:{PYTORCH_TRAIN_VERSION}-cpu-py311'

processor = ScriptProcessor(
    role=role,
    image_uri=image_uri,
    command=['python3'],
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    volume_size_in_gb=100,
    sagemaker_session=sess,
    base_job_name='discogs-build-data',
)

DATA_PREP_JOB_NAME = f'discogs-build-data-{datetime.now().strftime("%Y%m%d-%H%M%S")}'

print(f'Starting processing job: {DATA_PREP_JOB_NAME}')
print(f'  Instance: {PROCESSING_INSTANCE_TYPE}')
print(f'  Releases: {RELEASE_COUNT}')
print(f'  Output:   s3://{BUCKET_NAME}/{DATA_S3_PREFIX}/')

processor.run(
    code=str(repo / 'backend' / 'scripts' / 'processing_build_data.py'),
    source_dir=str(repo / 'backend'),
    dependencies=[str(repo / 'backend' / 'requirements-processing.txt')],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{BUCKET_NAME}/{DATA_S3_PREFIX}/processing-output/',
            output_name='output',
        ),
    ],
    arguments=[
        '--s3-bucket', BUCKET_NAME,
        '--s3-prefix', DATA_S3_PREFIX,
        '--count', str(RELEASE_COUNT),
        '--region', REGION,
    ],
    environment={'DISCOGS_USER_TOKEN': DISCOGS_TOKEN},
    job_name=DATA_PREP_JOB_NAME,
    wait=False,
)

print('\n✓ Processing job started (Space can stop — job keeps running)')
print(f'Monitor: SageMaker → Processing jobs → {DATA_PREP_JOB_NAME}')

## Step 3b: Check processing job status

Re-run after the job has been running. Wait for **Completed** before training.

In [ ]:
sm_client = boto3.client('sagemaker', region_name=REGION)

if 'DATA_PREP_JOB_NAME' not in dir():
    raise NameError('Run the processing job cell first (sets DATA_PREP_JOB_NAME)')

job = sm_client.describe_processing_job(ProcessingJobName=DATA_PREP_JOB_NAME)
status = job['ProcessingJobStatus']
print(f'Job: {DATA_PREP_JOB_NAME}')
print(f'Status: {status}')

if status == 'Completed':
    resp = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f'{DATA_S3_PREFIX}/images/', MaxKeys=5)
    n = resp.get('KeyCount', 0)
    print(f'\n✓ Data on s3://{BUCKET_NAME}/{DATA_S3_PREFIX}/')
    print(f'  Sample image keys: {n} listed (may be more)')
elif status == 'Failed':
    print('Failure reason:', job.get('FailureReason', 'see CloudWatch logs'))
else:
    print('Still running — re-run this cell in a few minutes')

## Step 4: Upload training code to S3

Data is already on S3 from the processing job. This uploads `sourcedir.tar.gz` for training.

In [ ]:
import os
import tarfile
import tempfile
import subprocess
from pathlib import Path

# Package train.py + ml/ for SageMaker (data already on S3 from processing job)
repo = Path(REPO_DIR)
if not repo.exists():
    raise FileNotFoundError(f"Clone repo to {REPO_DIR} first: git clone <url> {REPO_DIR}")

os.chdir(repo)
subprocess.run(
    ["bash", "prepare_for_studio.sh", BUCKET_NAME, REGION],
    check=True,
)
print(f"✓ Code at s3://{BUCKET_NAME}/code/ (data from processing job at s3://{BUCKET_NAME}/{DATA_S3_PREFIX}/)")

In [ ]:
print("="*50)
print("STEP 4: Training")
print("="*50)

sess = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))

estimator = PyTorch(
    entry_point='train.py',
    source_dir=BACKEND_DIR,
    role=role,
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    use_spot_instances=USE_SPOT_TRAINING,
    max_wait=86400 if USE_SPOT_TRAINING else None,
    checkpoint_s3_uri=f's3://{BUCKET_NAME}/checkpoints/' if USE_SPOT_TRAINING else None,
    framework_version=PYTORCH_TRAIN_VERSION,
    py_version='py311',
    hyperparameters={
        'epochs': '10',
        'batch-size': '16',
        'learning-rate': '0.001',
    },
    output_path=f's3://{BUCKET_NAME}/models/',
    sagemaker_session=sess,
    base_job_name='album-classifier',
    max_run=86400,
)

print("\nStarting training job...")
print(f"  Instance: ml.m5.xlarge")
print(f"  Data: s3://{BUCKET_NAME}/data/")
print(f"  Output: s3://{BUCKET_NAME}/models/")
print("\nThis will take 10-30 minutes...")

# Start training
estimator.fit({'training': f's3://{BUCKET_NAME}/data/'})

print(f"\n✓ Training complete!")
print(f"Model artifacts: {estimator.model_data}")

In [ ]:
print("="*50)
print("STEP 2: Deployment")
print("="*50)

# Create model from training job
model = PyTorchModel(
    model_data=estimator.model_data,
    role=role,
    entry_point='inference.py',
    framework_version=PYTORCH_INFER_VERSION,  # inference image map stops at 2.3
    py_version='py311',
    source_dir=BACKEND_DIR,
    sagemaker_session=sess,
)

print("\nDeploying endpoint...")
print("This will take 5-10 minutes...")

# Deploy endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=ENDPOINT_INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    wait=True  # Wait for deployment to complete
)

print(f"\n✓ Endpoint deployed: {ENDPOINT_NAME}")
print(f"Endpoint ARN: {predictor.endpoint}")

## Step 5: Test the Endpoint

Test your deployed endpoint with a sample image.

In [ ]:
print("="*50)
print("STEP 3: Testing")
print("="*50)

# Option 1: Download test image from S3
# Uncomment the line below to download an image
# !aws s3 cp s3://{BUCKET_NAME}/data/images/0.jpg test_image.jpg

try:
    # Read image file
    with open('test_image.jpg', 'rb') as f:
        image_data = f.read()
    
    # Make prediction
    print("\nMaking prediction...")
    response = predictor.predict(
        image_data,
        initial_args={'ContentType': 'image/jpeg'}
    )
    
    print("\nPredictions:")
    print(json.dumps(response, indent=2))
    
except FileNotFoundError:
    print("\n⚠ Test image not found.")
    print("To test:")
    print(f"  1. Download an image: !aws s3 cp s3://{BUCKET_NAME}/data/images/0.jpg test_image.jpg")
    print("  2. Or upload your own test image")
    print("  3. Then run this cell again")
except Exception as e:
    print(f"\n✗ Error testing endpoint: {e}")

## Step 6: Cleanup (Optional)

**Important**: Endpoints cost money while running. Delete when not in use!

In [ ]:
# Uncomment to delete endpoint when done
# predictor.delete_endpoint()
# predictor.delete_model()
# print("✓ Endpoint deleted")

## Next Steps

After deployment:

1. ✅ Test endpoint (Step 5 above)
2. ✅ Deploy API Gateway + Lambda (see `docs/DEPLOY_INFERENCE.md`)
3. ✅ Configure frontend (see `docs/FRONTEND_API_GATEWAY_SETUP.md`)

## Troubleshooting

- **Training fails**: Check CloudWatch logs in SageMaker console
- **Endpoint fails**: Verify model artifacts exist in S3
- **Access denied**: Check IAM role has S3 permissions

See `docs/STUDIO_WALKTHROUGH.md` for detailed troubleshooting.